In [ ]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
crime = pd.read_csv('/content/drive/MyDrive/Dataset/Origin/crime_dataset.csv')
print(crime.columns)

Index(['자치구별(1)', '자치구별(2)', '2024', '2024.1', '2024.2', '2024.3', '2024.4',
       '2024.5', '2024.6', '2024.7', '2024.8', '2024.9', '2024.10', '2024.11'],
      dtype='object')


In [ ]:
crime = crime.iloc[3:].copy()

crime.columns = [
    '지역구분',
    '자치구명',
    'total_occurrence',
    'total_clearance',
    'murder_occurrence',
    'murder_clearance',
    'robbery_occurrence',
    'robbery_clearance',
    'sexual_occurrence',
    'sexual_clearance',
    'theft_occurrence',
    'theft_clearance',
    'violence_occurrence',
    'violence_clearance'
]

#숫자형 변환
num_cols = crime.columns[2:]

for col in num_cols:
    crime[col] = pd.to_numeric(crime[col],errors='coerce')

#서울시 전체 행 제거
crime = crime[crime['자치구명'] != '소계']

crime.reset_index(drop=True, inplace=True)
crime.head()

,지역구분,자치구명,total_occurrence,total_clearance,murder_occurrence,murder_clearance,robbery_occurrence,robbery_clearance,sexual_occurrence,sexual_clearance,theft_occurrence,theft_clearance,violence_occurrence,violence_clearance
0,합계,종로구,2765,3248,2,3,5.0,10.0,209,933,1183,1056,1366,1246
1,합계,중구,2955,2180,5,4,5.0,5.0,177,121,1398,816,1370,1234
2,합계,용산구,3322,2475,10,6,4.0,3.0,267,235,1082,589,1959,1642
3,합계,성동구,2117,1510,5,6,3.0,3.0,114,90,966,544,1029,867
4,합계,광진구,2870,2219,2,2,6.0,7.0,235,183,1320,942,1307,1085


In [ ]:
#서울시 전체 범죄 발생수
seoul_total = crime['total_occurrence'].sum()

#crime_rate
crime['crime_rate'] = (crime['total_occurrence'] / seoul_total)

#clearance_rate
crime['clearance_rate'] = (crime['total_clearance'] / crime['total_occurrence'])

#Q1, Q3
q1 = crime['crime_rate'].quantile(0.25)
q3 = crime['crime_rate'].quantile(0.75)

def classify(x):
    if x <= q1:
        return "SAFE"

    elif x <= q3:
        return "NORMAL"

    else:
        return "DANGER"

crime['crime_grade'] = (crime['crime_rate'].apply(classify))

crime[['자치구명',
        'crime_rate',
        'clearance_rate',
        'crime_grade']]

,자치구명,crime_rate,clearance_rate,crime_grade
0,종로구,0.034212,1.174684,NORMAL
1,중구,0.036563,0.737733,NORMAL
2,용산구,0.041104,0.745033,NORMAL
3,성동구,0.026194,0.713274,SAFE
4,광진구,0.035511,0.773171,NORMAL
5,동대문구,0.039793,0.771766,NORMAL
6,중랑구,0.039211,0.777217,NORMAL
7,성북구,0.027023,0.800824,SAFE
8,강북구,0.028323,0.900393,SAFE
9,도봉구,0.021542,0.792074,SAFE


In [ ]:
crime.to_csv('/content/drive/MyDrive/Dataset/Preprocessed/crime_grade.csv', index=False)